# Figures from released results

Reads only the JSON files under `results/` (no data, no training) and draws the result figures.
Run from the repository root after `pip install -e .`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "results").is_dir() else Path.cwd().parent
RESULTS = ROOT / "results"
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## Table 1: LSP classifiers (intra-subject split, 81 test takes)

In [ ]:
rows = []
for name, label in [("bilstm_raw", "Bi-LSTM (landmarks)"), ("bilstm_enriched", "Bi-LSTM (+bones+motion)"),
                    ("stgcn", "ST-GCN"), ("hagcn", "HA-GCN")]:
    m = json.loads((RESULTS / "lsp" / name / "metrics.json").read_text())
    rows.append((label, 100 * m["test_metrics"]["accuracy"], m["num_params"]))
    print(f"{label:26s} acc={rows[-1][1]:.2f}%  params={rows[-1][2]:,}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh([r[0] for r in rows], [r[1] for r in rows], color="#4c78a8")
ax.set_xlim(60, 100)
ax.set_xlabel("test accuracy (%), intra-subject split")
fig.tight_layout()
fig.savefig(FIGURES / "table1_lsp_accuracy.png", dpi=200)

## Table 2: LSA64 few-shot on unseen signers (3 signer splits)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
for enc, color in [("stgcn", "#f58518"), ("hagcn", "#4c78a8")]:
    agg = json.loads((RESULTS / "lsa64" / enc / "summary.json").read_text())["aggregate"]
    ks = sorted(int(k) for k in agg["protonet"])
    for method, style in [("protonet", "-o"), ("finetune_k", "--s")]:
        mean = [100 * agg[method][str(k)]["mean"] for k in ks]
        std = [100 * agg[method][str(k)]["std"] for k in ks]
        ax.errorbar(ks, mean, yerr=std, fmt=style, color=color, capsize=3, label=f"{enc} {method}")
ax.set_xlabel("K (calibration takes per class)")
ax.set_ylabel("accuracy (%)")
ax.set_xticks(ks)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "table2_lsa64_accuracy_vs_k.png", dpi=200)

## Table 3: LSA64 -> LSP transfer (HA-GCN encoders, one per LSA64 signer split)

`corrected_3_splits.json` is produced by `scripts/reaggregate_transfer.py` from the original per-trial results.

In [ ]:
data = json.loads((RESULTS / "transfer" / "corrected_3_splits.json").read_text())
fig, ax = plt.subplots(figsize=(5.5, 4))
for sid, color in [("signer_00", "#4c78a8"), ("signer_01", "#e45756")]:
    r = data["signers"][sid]
    for method, style in [("protonet", "-o"), ("finetune_k", "--s")]:
        mean = [100 * r["aggregate"][method][str(k)]["mean"] for k in r["ks"]]
        std = [100 * r["aggregate"][method][str(k)]["std"] for k in r["ks"]]
        ax.errorbar(r["ks"], mean, yerr=std, fmt=style, color=color, capsize=3, label=f"{sid} {method}")
ax.axhline(100 / 30, color="grey", linewidth=0.8, linestyle=":", label="chance (30 classes)")
ax.set_xlabel("K (calibration takes per class)")
ax.set_ylabel("accuracy (%)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "table3_transfer_accuracy_vs_k.png", dpi=200)